# Neuron Overlap: Experiment 1 (Incentive Core) vs Experiment 3 (Anhedonic)

Cross-references three neuron sets from Exp1 against the top changed neurons from Exp3:
- `master_incentive_core.csv` — neurons universal across ALL incentive conditions (3528 neurons)
- `universal_money_neurons.csv` — money-sensitive neurons (5467)
- `universal_reward_neurons.csv` — reward-sensitive neurons (5558)

Question: **do the neurons that fire for incentive/reward in Exp1 overlap with the neurons most suppressed or amplified by the anhedonic prompt in Exp3?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib_venn import venn2, venn3

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size']  = 11

In [ ]:
# ── Load Exp1 neuron sets ─────────────────────────────────────────────────────
df_core   = pd.read_csv('data/master_incentive_core.csv')
df_money  = pd.read_csv('data/universal_money_neurons.csv')
df_reward = pd.read_csv('data/universal_reward_neurons.csv')

# Convert to sets of (layer, neuron) tuples
set_core   = set(zip(df_core['layer'],   df_core['neuron']))
set_money  = set(zip(df_money['layer'],  df_money['neuron']))
set_reward = set(zip(df_reward['layer'], df_reward['neuron']))

print(f'Exp1 master_incentive_core   : {len(set_core):>5} neurons')
print(f'Exp1 universal_money         : {len(set_money):>5} neurons')
print(f'Exp1 universal_reward        : {len(set_reward):>5} neurons')
print(f'Total unique neurons in space: 28 × 18944 = {28*18944:,}')

In [ ]:
# ── Load Exp3 neuron activations ──────────────────────────────────────────────
neu_normal    = np.load('activations/activations_neurons_normal.npy')    # (100, 28, 18944)
neu_anhedonic = np.load('activations/activations_neurons_anhedonic.npy')

n_prompts, n_layers, n_neurons = neu_normal.shape

mean_normal    = neu_normal.mean(axis=0)    # (28, 18944)
mean_anhedonic = neu_anhedonic.mean(axis=0)
diff_signed    = mean_anhedonic - mean_normal
diff_abs       = np.abs(diff_signed)

# Effect size
pooled_std  = np.sqrt((neu_normal.std(axis=0)**2 + neu_anhedonic.std(axis=0)**2) / 2 + 1e-8)
effect_size = diff_signed / pooled_std

# Build Exp3 top-N sets at various thresholds
for topn in [50, 100, 200, 500]:
    flat_idx = np.argsort(diff_abs.flatten())[::-1][:topn]
    tl = flat_idx // n_neurons
    tn = flat_idx %  n_neurons
    s  = set(zip(tl, tn))
    overlap_core   = len(s & set_core)
    overlap_money  = len(s & set_money)
    overlap_reward = len(s & set_reward)
    print(f'Exp3 top-{topn:>4} | core: {overlap_core:>3}  money: {overlap_money:>3}  reward: {overlap_reward:>3}')

print('\nBaseline (random chance):')
total = n_layers * n_neurons
for name, s_exp1 in [('core', set_core), ('money', set_money), ('reward', set_reward)]:
    for topn in [50, 200]:
        expected = topn * len(s_exp1) / total
        print(f'  top-{topn} ∩ {name}: expected {expected:.2f} by chance')

In [ ]:
# ── Build Exp3 top sets for detailed analysis ─────────────────────────────────
TOP_N = 500
flat_idx   = np.argsort(diff_abs.flatten())[::-1][:TOP_N]
top_layers = flat_idx // n_neurons
top_neurs  = flat_idx %  n_neurons

set_exp3 = set(zip(top_layers, top_neurs))

# Separate into UP and DOWN
set_exp3_up   = {(l,n) for l,n in set_exp3 if diff_signed[l,n] > 0}
set_exp3_down = {(l,n) for l,n in set_exp3 if diff_signed[l,n] < 0}

print(f'Exp3 top-{TOP_N}: {len(set_exp3_up)} UP, {len(set_exp3_down)} DOWN')

for name, s1 in [('core', set_core), ('money', set_money), ('reward', set_reward)]:
    all_o  = set_exp3 & s1
    up_o   = set_exp3_up & s1
    down_o = set_exp3_down & s1
    print(f'\nExp3 top-{TOP_N} ∩ {name}:')
    print(f'  Total overlap : {len(all_o)}')
    print(f'  UP   (↑ anhed): {len(up_o)}')
    print(f'  DOWN (↓ anhed): {len(down_o)}')

## Plot 1: Overlap Bar Chart — UP vs DOWN vs Exp1 Sets

In [ ]:
exp1_sets   = [('Incentive Core\n(3528)', set_core),
               ('Money\n(5467)',           set_money),
               ('Reward\n(5558)',          set_reward)]
top_ns      = [50, 100, 200, 500]
colors_up   = '#D32F2F'
colors_down = '#1565C0'

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (exp1_name, exp1_set) in zip(axes, exp1_sets):
    overlaps_up   = []
    overlaps_down = []
    expected      = []

    for topn in top_ns:
        flat_idx = np.argsort(diff_abs.flatten())[::-1][:topn]
        tl = flat_idx // n_neurons
        tn = flat_idx %  n_neurons
        s  = set(zip(tl, tn))
        s_up   = {(l,n) for l,n in s if diff_signed[l,n] > 0}
        s_down = {(l,n) for l,n in s if diff_signed[l,n] < 0}
        overlaps_up.append(len(s_up & exp1_set))
        overlaps_down.append(len(s_down & exp1_set))
        expected.append(topn * len(exp1_set) / (n_layers * n_neurons))

    x = np.arange(len(top_ns))
    w = 0.3
    ax.bar(x - w/2, overlaps_up,   w, label='↑ UP (anhedonic)',   color=colors_up,   alpha=0.85)
    ax.bar(x + w/2, overlaps_down, w, label='↓ DOWN (anhedonic)', color=colors_down, alpha=0.85)
    ax.plot(x, expected, 'k--o', ms=5, label='Random chance', linewidth=1.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Top {n}' for n in top_ns])
    ax.set_ylabel('# Overlapping neurons')
    ax.set_title(f'Exp3 ∩ Exp1 {exp1_name}', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Overlap Between Exp3 Anhedonic-Changed Neurons and Exp1 Incentive Neurons\n'
             'Dashed line = expected by random chance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_overlap_bar.png', bbox_inches='tight')
plt.show()

## Plot 2: Layer Profile of Overlapping Neurons

In [ ]:
# For each layer: how many of the top-500 Exp3 neurons are also in Exp1 sets?
flat_idx   = np.argsort(diff_abs.flatten())[::-1][:500]
top_layers = flat_idx // n_neurons
top_neurs  = flat_idx %  n_neurons
top_signs  = diff_signed[top_layers, top_neurs]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (exp1_name, exp1_set) in zip(axes, exp1_sets):
    up_per_layer   = np.zeros(n_layers)
    down_per_layer = np.zeros(n_layers)

    for l, n, d in zip(top_layers, top_neurs, top_signs):
        if (l, n) in exp1_set:
            if d > 0: up_per_layer[l]   += 1
            else:     down_per_layer[l] += 1

    x = np.arange(n_layers)
    ax.bar(x,  up_per_layer,   label='↑ UP',   color='#D32F2F', alpha=0.8)
    ax.bar(x, -down_per_layer, label='↓ DOWN', color='#1565C0', alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Layer')
    ax.set_ylabel('# Overlapping neurons')
    ax.set_title(f'Exp3 top-500 ∩ Exp1 {exp1_name}\nper layer', fontsize=10)
    ax.set_xticks(x)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Which Layers Concentrate the Overlapping Neurons?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_overlap_by_layer.png', bbox_inches='tight')
plt.show()

## Plot 3: Activation Profiles of Overlapping Neurons (Violin)

In [ ]:
# Get the actual overlapping neurons between Exp3 top-500 and Exp1 core
flat_idx   = np.argsort(diff_abs.flatten())[::-1][:500]
top_layers = flat_idx // n_neurons
top_neurs  = flat_idx %  n_neurons
set_exp3   = set(zip(top_layers, top_neurs))

overlap_core = sorted(set_exp3 & set_core, 
                      key=lambda x: diff_abs[x[0], x[1]], reverse=True)

print(f'Overlapping neurons (Exp3 top-500 ∩ Exp1 core): {len(overlap_core)}')
for l, n in overlap_core[:20]:
    d = diff_signed[l, n]
    e = effect_size[l, n]
    print(f'  Layer {l:2d}  Neuron {n:5d}  Δ={d:+.4f}  d={e:+.3f}  {"↑" if d>0 else "↓"}')

In [ ]:
# Plot violin for top overlapping neurons
PLOT_N = min(12, len(overlap_core))

if PLOT_N == 0:
    print('No overlap found in top-500 — try increasing TOP_N above.')
else:
    neu_neutral = np.load('activations/activations_neurons_neutral.npy')

    fig, axes = plt.subplots(3, 4, figsize=(18, 12))
    axes = axes.flatten()

    for i, (l, n) in enumerate(overlap_core[:PLOT_N]):
        ax = axes[i]
        vals = [neu_normal[:, l, n], neu_anhedonic[:, l, n], neu_neutral[:, l, n]]
        parts = ax.violinplot(vals, positions=[0,1,2], showmeans=True, showmedians=False)
        for pc, color in zip(parts['bodies'], ['#2196F3','#F44336','#4CAF50']):
            pc.set_facecolor(color)
            pc.set_alpha(0.7)
        ax.set_xticks([0,1,2])
        ax.set_xticklabels(['Normal','Anhedonic','Neutral'], fontsize=8)
        d = diff_signed[l, n]
        e = effect_size[l, n]
        ax.set_title(f'Layer {l}  ·  Neuron {n}\nΔ={d:+.3f}   d={e:+.2f}   {"↑" if d>0 else "↓"}', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        for pos, v, c in [(0,vals[0],'#2196F3'),(1,vals[1],'#F44336'),(2,vals[2],'#4CAF50')]:
            ax.text(pos, ax.get_ylim()[1]*0.97, f'{v.mean():.3f}', ha='center', fontsize=7, color=c, fontweight='bold')

    # Hide unused axes
    for j in range(PLOT_N, 12):
        axes[j].set_visible(False)

    plt.suptitle('Activation Profiles: Neurons in BOTH Exp1 Incentive Core AND Exp3 Top-500\n'
                 'Blue=Normal  Red=Anhedonic  Green=Neutral', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig_overlap_violin.png', bbox_inches='tight')
    plt.show()

## Plot 4: Scatter — Effect Size vs Δ Mean, Colored by Exp1 Membership

In [ ]:
# For all top-500 Exp3 neurons: scatter effect size vs delta, color by Exp1 set membership
flat_idx   = np.argsort(diff_abs.flatten())[::-1][:500]
top_layers = flat_idx // n_neurons
top_neurs  = flat_idx %  n_neurons

deltas  = diff_signed[top_layers, top_neurs]
effects = effect_size[top_layers, top_neurs]

in_core   = np.array([(l,n) in set_core   for l,n in zip(top_layers, top_neurs)])
in_money  = np.array([(l,n) in set_money  for l,n in zip(top_layers, top_neurs)])
in_reward = np.array([(l,n) in set_reward for l,n in zip(top_layers, top_neurs)])
in_any    = in_core | in_money | in_reward

fig, ax = plt.subplots(figsize=(10, 7))

# Background: not in any Exp1 set
ax.scatter(deltas[~in_any], effects[~in_any], c='#BDBDBD', s=25, alpha=0.5, label='Not in Exp1', zorder=1)
# In money only
mask = in_money & ~in_core & ~in_reward
ax.scatter(deltas[mask], effects[mask], c='#FF9800', s=60, alpha=0.8, label='Exp1 Money only', zorder=2)
# In reward only
mask = in_reward & ~in_core & ~in_money
ax.scatter(deltas[mask], effects[mask], c='#9C27B0', s=60, alpha=0.8, label='Exp1 Reward only', zorder=2)
# In core (incentive universal)
ax.scatter(deltas[in_core], effects[in_core], c='#F44336', s=100, alpha=0.9,
           label=f'Exp1 Incentive Core ({in_core.sum()})', edgecolors='black', linewidth=0.5, zorder=3)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Δ Mean Activation (Anhedonic − Normal)', fontsize=12)
ax.set_ylabel('Effect Size (Cohen\'s d)', fontsize=12)
ax.set_title('Exp3 Top-500 Neurons: Effect Size vs Δ Activation\nColored by Exp1 Set Membership', fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('fig_scatter_overlap.png', bbox_inches='tight')
plt.show()

print(f'In core   : {in_core.sum()}')
print(f'In money  : {in_money.sum()}')
print(f'In reward : {in_reward.sum()}')
print(f'In any    : {in_any.sum()}')

## Final Table: Overlapping Neurons with Full Stats

In [ ]:
flat_idx   = np.argsort(diff_abs.flatten())[::-1][:500]
top_layers = flat_idx // n_neurons
top_neurs  = flat_idx %  n_neurons

rows = []
for rank, (l, n) in enumerate(zip(top_layers, top_neurs), 1):
    in_c = (l,n) in set_core
    in_m = (l,n) in set_money
    in_r = (l,n) in set_reward
    if in_c or in_m or in_r:
        rows.append({
            'exp3_rank':     rank,
            'layer':         l,
            'neuron':        n,
            'delta':         round(float(diff_signed[l,n]), 4),
            'effect_size':   round(float(effect_size[l,n]),  4),
            'direction':     'UP' if diff_signed[l,n] > 0 else 'DOWN',
            'in_core':       in_c,
            'in_money':      in_m,
            'in_reward':     in_r,
        })

df_overlap = pd.DataFrame(rows).sort_values('exp3_rank')
print(f'Total overlapping neurons: {len(df_overlap)}')
print(df_overlap.to_string(index=False))

df_overlap.to_csv('overlap_exp1_exp3.csv', index=False)
print('\nSaved → overlap_exp1_exp3.csv')